# Charged particles in a magnetic bottle

A magnetic mirror traps charges because $\mu = m v_\perp^2 / 2B$ (the magnetic moment of the gyration) is an
adiabatic invariant: as a particle spirals into a stronger field, $v_\perp$ grows, and since the speed is fixed its
motion along the field stops and reverses. A particle with pitch angle $\alpha$ (between $\mathbf v$ and $\mathbf B$)
at the field minimum $B_0$ turns around where $B = B_0 / \sin^2\alpha$.

The field here is $B_z = B_0 (1 + z^2/L^2)$ with the radial component that keeps it divergence-free, given to
`FieldForce` as a Python function. The `boris` integrator rotates velocities about $\mathbf B$ exactly, so the speed
is conserved to round-off. The predicted mirror point is $z = L\cot\alpha$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import physim as ps

In [ ]:
B0, L = 1.0, 1.0

def bottle(t, pos):
    x, y, z = pos[:, 0], pos[:, 1], pos[:, 2]
    return np.stack([-B0 * x * z / L**2, -B0 * y * z / L**2, B0 * (1 + z**2 / L**2)], axis=1)

alphas = np.radians(np.linspace(20, 80, 25))
v = 0.05                                     # gyroradius v sin(alpha) / B0 << L
w = ps.World("boris")
for a in alphas:                             # guiding centres on the axis
    w.add_particle([v * np.sin(a) / B0, 0, 0], [0, -v * np.sin(a), v * np.cos(a)], charge=1.0)
w.add_force(ps.FieldForce(B=bottle))
tr = w.run(0.05, 20_000, record_every=20, energies=False)

speed = np.linalg.norm(tr.vel, axis=2)
B = np.stack([bottle(0, p) for p in tr.pos])
Bmag = np.linalg.norm(B, axis=2)
v_par = np.einsum("fni,fni->fn", tr.vel, B) / Bmag
mu = (speed**2 - v_par**2) / (2 * Bmag)
z_max = np.abs(tr.pos[:, :, 2]).max(axis=0)
print(f"speed conserved to {np.abs(speed / speed[0] - 1).max():.1e}; "
      f"mu varies by {np.max(mu.std(axis=0) / mu.mean(axis=0)):.1e} (relative)")
print(f"mirror points match L cot(alpha) to {np.max(np.abs(z_max / (L / np.tan(alphas)) - 1)):.1e}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
for k, i in enumerate([2, 10, 20]):
    a1.plot(tr.t, tr.pos[:, i, 2], color=ps.plot.color(k), lw=1, label=rf"$\alpha$ = {np.degrees(alphas[i]):.0f}°")
a1.set_xlabel("t"); a1.set_ylabel("z (along the field)"); ps.plot.tidy(a1)
aa = np.linspace(15, 85, 200)
a2.plot(aa, L / np.tan(np.radians(aa)), color=ps.plot.color(2), lw=1.5, label=r"adiabatic theory $L\cot\alpha$")
a2.plot(np.degrees(alphas), z_max, "o", color=ps.plot.color(0), ms=5, label="simulated")
a2.set_xlabel("pitch angle at the centre (deg)"); a2.set_ylabel("mirror point"); ps.plot.tidy(a2, grid=True);